# Bird Song Generation
## Stage 3: Residual CNN Species Classifier

This notebook trains and evaluates the supervised classifier for American Robin, Northern Cardinal, and Song Sparrow. It follows the same sectioned, explanation-first style as the Assignment 3 notebook, while importing the project implementation instead of duplicating pipeline code.

### Loading Modules

We load the scientific Python tools plus the reusable preprocessing, dataset, model, and checkpoint helpers from the project source package.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from torch import nn

# Make the notebook work when opened from either the repository root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from bird_song.classifier.model import BirdSongCNN
from bird_song.config import SpectrogramConfig
from bird_song.data import ManifestDataset, make_loader, resolve_dataset_root
from bird_song.runtime import atomic_torch_save, load_checkpoint, seed_everything

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Project root:', PROJECT_ROOT)
print('Device:', device)

### Project Configuration

The shared Stage 2 configuration converts every clip to a normalized 128 by 128 log-mel spectrogram. Set RUN_TRAINING to True only when you want to train a new checkpoint; by default, the notebook evaluates the supplied checkpoint.

In [ ]:
DATASET_ROOT = PROJECT_ROOT / 'bird_songs_dataset'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'spectrogram.json'
TRAIN_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_train.csv'
VALIDATION_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_validation.csv'
TEST_MANIFEST = PROJECT_ROOT / 'manifests' / 'full_dataset_test.csv'
CHECKPOINT_PATH = PROJECT_ROOT / 'classifier_artifacts' / 'Harvey_classifier' / 'best.pt'
RUN_DIR = PROJECT_ROOT / 'runs' / 'notebook_classifier'

SEED = 42
BATCH_SIZE = 64
WORKERS = 0  # A notebook-friendly choice; increase after confirming your local setup.
EPOCHS = 40
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
PATIENCE = 8
RUN_TRAINING = False

seed_everything(SEED)
dataset_root = resolve_dataset_root(PROJECT_ROOT, DATASET_ROOT)
spectrogram_config = SpectrogramConfig.from_json(CONFIG_PATH)
classes = tuple(sorted(pd.read_csv(TRAIN_MANIFEST)['name'].unique()))

split_summary = pd.concat([
    pd.read_csv(TRAIN_MANIFEST).assign(split='train'),
    pd.read_csv(VALIDATION_MANIFEST).assign(split='validation'),
    pd.read_csv(TEST_MANIFEST).assign(split='test'),
])[['split', 'name']].value_counts().rename('clips').reset_index()
display(split_summary)
print('Classes:', classes)
print('Log-mel shape:', (spectrogram_config.n_mels, spectrogram_config.spectrogram_width))

### Dataset and Data Loaders

The training set uses waveform and spectrogram augmentation. Validation and test clips use deterministic preprocessing. All clips from an original recording remain in only one split.

In [ ]:
train_set = ManifestDataset(TRAIN_MANIFEST, dataset_root, classes, spectrogram_config, training=True)
validation_set = ManifestDataset(VALIDATION_MANIFEST, dataset_root, classes, spectrogram_config, training=False)
test_set = ManifestDataset(TEST_MANIFEST, dataset_root, classes, spectrogram_config, training=False)

train_loader = make_loader(train_set, BATCH_SIZE, WORKERS, training=True)
validation_loader = make_loader(validation_set, BATCH_SIZE, WORKERS)
test_loader = make_loader(test_set, BATCH_SIZE, WORKERS)

print(f'Train: {len(train_set):,} clips | Validation: {len(validation_set):,} clips | Test: {len(test_set):,} clips')

#### A Log-mel Batch

Each model input has one channel and a fixed 128 by 128 time-frequency representation. The training view below may include augmentation.

In [ ]:
specs, labels, paths = next(iter(train_loader))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
for axis, spec, label in zip(axes, specs[:3], labels[:3]):
    image = axis.imshow(spec.squeeze(0), origin='lower', aspect='auto', cmap='magma', vmin=-1, vmax=1)
    axis.set_title(classes[label])
    axis.set_xlabel('Time frame')
axes[0].set_ylabel('Mel bin')
fig.colorbar(image, ax=axes, label='Normalized log-mel')
plt.show()
print('Batch shape:', tuple(specs.shape))
print('First file:', paths[0])

### ResNet-style Classifier

The classifier has a convolutional stem followed by six residual blocks. Each block has two 3 by 3 convolutions; three skip paths add a 1 by 1 projection for downsampling or channel changes. Average and maximum pooling are concatenated before the two-layer classification head.

128 by 128 input -> stem -> 6 residual blocks -> average/max pooling -> 512 -> 128 -> 3 classes

In [ ]:
model = BirdSongCNN(num_classes=len(classes)).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(model)
print(f'\nTrainable parameters: {parameter_count:,}')
with torch.inference_mode():
    logits = model(specs[:2].to(device))
print('Logit shape:', tuple(logits.shape))

### Training Loop

We optimize cross-entropy with light label smoothing, AdamW, cosine learning-rate decay, gradient clipping, and early stopping based on validation accuracy. This mirrors the command-line Stage 3 training workflow.

In [ ]:
@torch.inference_mode()
def evaluate_epoch(model, loader, criterion):
    model.eval()
    total_loss = total_correct = total_items = 0
    for batch_specs, batch_labels, _ in loader:
        batch_specs = batch_specs.to(device)
        batch_labels = batch_labels.to(device)
        batch_logits = model(batch_specs)
        total_loss += criterion(batch_logits, batch_labels).item() * batch_labels.numel()
        total_correct += (batch_logits.argmax(1) == batch_labels).sum().item()
        total_items += batch_labels.numel()
    return total_loss / total_items, total_correct / total_items

def train_classifier(model, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_state, best_accuracy, stale_epochs = None, float('-inf'), 0
    history = {'epoch': [], 'train_loss': [], 'validation_loss': [], 'validation_accuracy': []}

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = item_count = 0
        for batch_specs, batch_labels, _ in train_loader:
            batch_specs = batch_specs.to(device)
            batch_labels = batch_labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_specs), batch_labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            running_loss += loss.item() * batch_labels.numel()
            item_count += batch_labels.numel()

        validation_loss, validation_accuracy = evaluate_epoch(model, validation_loader, criterion)
        history['epoch'].append(epoch)
        history['train_loss'].append(running_loss / item_count)
        history['validation_loss'].append(validation_loss)
        history['validation_accuracy'].append(validation_accuracy)
        print(f'Epoch {epoch:02d}/{epochs} | train loss: {history["train_loss"][-1]:.4f} | val loss: {validation_loss:.4f} | val accuracy: {validation_accuracy:.2%}')

        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE:
            print(f'Early stopping after epoch {epoch}.')
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_accuracy

#### Train a New Model (Optional)

Set RUN_TRAINING to True in the configuration cell to run the training cell. The result is stored in the runs directory, leaving the supplied reference checkpoint unchanged.

In [ ]:
if RUN_TRAINING:
    model = BirdSongCNN(num_classes=len(classes)).to(device)
    model, history, best_validation_accuracy = train_classifier(model)
    new_checkpoint = RUN_DIR / 'best.pt'
    atomic_torch_save({
        'format_version': 1,
        'model_state': model.state_dict(),
        'model_config': model.metadata(),
        'classes': list(classes),
        'spectrogram_config': spectrogram_config.to_dict(),
        'best_validation_accuracy': best_validation_accuracy,
    }, new_checkpoint)
    CHECKPOINT_PATH = new_checkpoint
    print('Saved checkpoint:', new_checkpoint)
else:
    print('Skipped training. Set RUN_TRAINING = True to train a new classifier.')

#### Loss and Validation Accuracy

When a new run has completed, plot the learning curves to check whether the model is still improving or has begun to overfit.

In [ ]:
if 'history' in globals():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(history['epoch'], history['train_loss'], label='Train')
    axes[0].plot(history['epoch'], history['validation_loss'], label='Validation')
    axes[0].set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Loss dynamics')
    axes[0].legend()
    axes[1].plot(history['epoch'], history['validation_accuracy'], color='tab:green')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Validation accuracy', ylim=(0, 1))
    plt.show()
else:
    print('No new training history yet; the supplied checkpoint is evaluated below.')

### Held-out Test Evaluation

The test split contains recordings unseen during training and validation. We load the selected checkpoint, calculate class probabilities for every test clip, and summarize the result with standard classification metrics and a confusion matrix.

In [ ]:
checkpoint_model, checkpoint_classes, checkpoint_config, checkpoint = load_checkpoint(CHECKPOINT_PATH, device)
assert checkpoint_classes == classes
assert checkpoint_config == spectrogram_config

predictions, targets, confidences = [], [], []
checkpoint_model.eval()
with torch.inference_mode():
    for batch_specs, batch_labels, _ in test_loader:
        probabilities = checkpoint_model(batch_specs.to(device)).softmax(1).cpu()
        predictions.extend(probabilities.argmax(1).tolist())
        targets.extend(batch_labels.tolist())
        confidences.extend(probabilities.max(1).values.tolist())

report = classification_report(targets, predictions, target_names=classes, output_dict=True, zero_division=0)
print(f'Checkpoint epoch: {checkpoint.get("epoch", "not recorded")}')
print(f'Test accuracy: {report["accuracy"]:.2%}')
print(f'Test macro F1: {report["macro avg"]["f1-score"]:.2%}')
display(pd.DataFrame(report).T.loc[list(classes) + ["macro avg", "weighted avg"]])

#### Confusion Matrix

Rows show the true species and columns show the predicted species. Off-diagonal cells identify the species pairs that the classifier most often confuses.

In [ ]:
matrix = confusion_matrix(targets, predictions, labels=range(len(classes)))
display_matrix = ConfusionMatrixDisplay(matrix, display_labels=classes)
fig, axis = plt.subplots(figsize=(7, 5.5))
display_matrix.plot(ax=axis, cmap='Blues', colorbar=False, values_format='d')
axis.set_title('Held-out test-set confusion matrix')
plt.xticks(rotation=20, ha='right')
plt.show()

print(f'Mean prediction confidence: {np.mean(confidences):.2%}')

### Findings

The supplied checkpoint was selected at epoch 8 with 88.25% validation accuracy. Its held-out test performance is recorded in the model card: 90.39% accuracy and 90.44% macro F1. This closed-set classifier is useful for Stage 6 target-species scores, but it cannot by itself determine whether generated audio is realistic.